# SpaceX Falcon 9 First Stage Landing Prediction — Machine Learning Prediction

Trains and compares four classifiers (Logistic Regression, SVM, Decision Tree, KNN) tuned via GridSearchCV(cv=10) to predict first-stage landing success.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix

data = pd.read_csv("data/dataset_part_2.csv")
X = pd.read_csv("data/dataset_part_3.csv")

print("data shape:", data.shape)
print("X shape:", X.shape)

# TASK 1
Y = data['Class'].to_numpy()
print("Y shape:", Y.shape)

# TASK 2
transform = preprocessing.StandardScaler()
X = transform.fit_transform(X)

# TASK 3
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=2)
print("X_train:", X_train.shape, "X_test:", X_test.shape, "Y_train:", Y_train.shape, "Y_test:", Y_test.shape)

results = {}

def save_confmat(y_true, y_pred, name, fname):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(3.6, 3.2))
    sns.heatmap(cm, annot=True, fmt='d', ax=ax, cmap="Blues")
    ax.set_xlabel('Predicted labels')
    ax.set_ylabel('True labels')
    ax.set_title(f'Confusion Matrix - {name}')
    ax.xaxis.set_ticklabels(['did not land', 'land'])
    ax.yaxis.set_ticklabels(['did not land', 'landed'])
    plt.tight_layout()
    plt.savefig(fname, dpi=150)
    plt.close(fig)
    tn, fp, fn, tp = cm.ravel()
    return {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}

# TASK 4: Logistic Regression
parameters = {'C': [0.01, 0.1, 1], 'penalty': ['l2'], 'solver': ['lbfgs']}
lr = LogisticRegression(max_iter=1000)
logreg_cv = GridSearchCV(lr, parameters, cv=10)
logreg_cv.fit(X_train, Y_train)
print("LR best params:", logreg_cv.best_params_)
print("LR best cv score:", logreg_cv.best_score_)

# TASK 5
lr_test_acc = logreg_cv.score(X_test, Y_test)
print("LR test acc:", lr_test_acc)
yhat_lr = logreg_cv.predict(X_test)
cm_lr = save_confmat(Y_test, yhat_lr, "Logistic Regression", "cm_lr.png")
print("LR cm:", cm_lr)

# TASK 6: SVM
parameters = {'kernel': ('linear', 'rbf', 'poly', 'sigmoid'),
              'C': np.logspace(-3, 3, 5),
              'gamma': np.logspace(-3, 3, 5)}
svm = SVC()
svm_cv = GridSearchCV(svm, parameters, cv=10)
svm_cv.fit(X_train, Y_train)
print("SVM best params:", svm_cv.best_params_)
print("SVM best cv score:", svm_cv.best_score_)

# TASK 7
svm_test_acc = svm_cv.score(X_test, Y_test)
print("SVM test acc:", svm_test_acc)
yhat_svm = svm_cv.predict(X_test)
cm_svm = save_confmat(Y_test, yhat_svm, "SVM", "cm_svm.png")
print("SVM cm:", cm_svm)

# TASK 8: Decision Tree
parameters = {'criterion': ['gini', 'entropy'],
              'splitter': ['best', 'random'],
              'max_depth': [2*n for n in range(1, 10)],
              'max_features': ['sqrt', 'log2'],
              'min_samples_leaf': [1, 2, 4],
              'min_samples_split': [2, 5, 10]}
tree = DecisionTreeClassifier(random_state=2)
tree_cv = GridSearchCV(tree, parameters, cv=10)
tree_cv.fit(X_train, Y_train)
print("Tree best params:", tree_cv.best_params_)
print("Tree best cv score:", tree_cv.best_score_)

# TASK 9
tree_test_acc = tree_cv.score(X_test, Y_test)
print("Tree test acc:", tree_test_acc)
yhat_tree = tree_cv.predict(X_test)
cm_tree = save_confmat(Y_test, yhat_tree, "Decision Tree", "cm_tree.png")
print("Tree cm:", cm_tree)

# TASK 10: KNN
parameters = {'n_neighbors': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
              'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'],
              'p': [1, 2]}
KNN = KNeighborsClassifier()
knn_cv = GridSearchCV(KNN, parameters, cv=10)
knn_cv.fit(X_train, Y_train)
print("KNN best params:", knn_cv.best_params_)
print("KNN best cv score:", knn_cv.best_score_)

# TASK 11
knn_test_acc = knn_cv.score(X_test, Y_test)
print("KNN test acc:", knn_test_acc)
yhat_knn = knn_cv.predict(X_test)
cm_knn = save_confmat(Y_test, yhat_knn, "KNN", "cm_knn.png")
print("KNN cm:", cm_knn)

# TASK 12: best method
algorithms = {'KNN': knn_cv.best_score_, 'Tree': tree_cv.best_score_,
              'LogisticRegression': logreg_cv.best_score_, 'SVM': svm_cv.best_score_}
best_algorithm = max(algorithms, key=algorithms.get)
print("Best algorithm (by CV score):", best_algorithm, algorithms)

test_accs = {'LogisticRegression': lr_test_acc, 'SVM': svm_test_acc, 'Tree': tree_test_acc, 'KNN': knn_test_acc}
best_test_algorithm = max(test_accs, key=test_accs.get)
print("Best algorithm (by TEST score):", best_test_algorithm, test_accs)

summary = {
    "n_total": int(X_train.shape[0] + X_test.shape[0]),
    "n_test": int(X_test.shape[0]),
    "n_train": int(X_train.shape[0]),
    "models": {
        "LogisticRegression": {
            "best_params": logreg_cv.best_params_,
            "cv_score": float(logreg_cv.best_score_),
            "test_score": float(lr_test_acc),
            "confusion_matrix": cm_lr,
        },
        "SVM": {
            "best_params": svm_cv.best_params_,
            "cv_score": float(svm_cv.best_score_),
            "test_score": float(svm_test_acc),
            "confusion_matrix": cm_svm,
        },
        "DecisionTree": {
            "best_params": tree_cv.best_params_,
            "cv_score": float(tree_cv.best_score_),
            "test_score": float(tree_test_acc),
            "confusion_matrix": cm_tree,
        },
        "KNN": {
            "best_params": knn_cv.best_params_,
            "cv_score": float(knn_cv.best_score_),
            "test_score": float(knn_test_acc),
            "confusion_matrix": cm_knn,
        },
    },
    "best_by_cv_score": best_algorithm,
    "best_by_test_score": best_test_algorithm,
}

with open("results.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\nSUMMARY JSON:")
print(json.dumps(summary, indent=2, default=str))


## Results Summary

- Dataset: 90 launches -> 72 train / 18 test (test_size=0.2, random_state=2)
- Logistic Regression: CV 82.14%, Test 83.33% (C=1, penalty=l2, solver=lbfgs)
- SVM: CV 84.82% (best of the four), Test 83.33% (kernel=sigmoid, C=1.0, gamma≈0.0316)
- Decision Tree: CV 83.39%, Test 72.22%
- KNN: CV 83.39%, Test 83.33% (n_neighbors=6, p=1)
- Best by cross-validation: SVM. Best by test accuracy: 3-way tie (Logistic Regression, SVM, KNN) at 83.33%.